Dataset personalizado de PyTorch.

Carga de secuencias y etiquetas.

Tokenización de transcripciones.

Padding de secuencias por batch.

DataLoader y collate_fn.

Validación de longitudes para CTC.

In [1]:
import json
import random
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader

In [2]:
# Rutas
CLIPS_DIR = Path(r"D:\ViSpeR\clips")
METADATA_PATH = CLIPS_DIR / "metadata.json"

# Reproducibilidad
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Configuración del procesamiento
IMAGE_MODE = None  # Pendiente: "grayscale" o "rgb"

# DataLoader
BATCH_SIZE = 4
NUM_WORKERS = 0  # Configuración inicial compatible con Windows

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("Dispositivo:", DEVICE)
print("Metadata:", METADATA_PATH)
print("Metadata existe:", METADATA_PATH.exists())

PyTorch: 2.6.0+cu126
CUDA disponible: True
Dispositivo: cuda
Metadata: D:\ViSpeR\clips\metadata.json
Metadata existe: True


In [3]:
with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

clips_df = pd.DataFrame([
    {
        "path": Path(path),
        "label": info["label"],
        "frames": info["length"],
    }
    for path, info in metadata.items()
])

print("Cantidad de clips:", len(clips_df))
display(clips_df.head())

Cantidad de clips: 1304


,path,label,frames
0,D:\ViSpeR\clips\videos\0YEeb-WLF2w\00000.mp4,La ficción es una herramienta poderosa,108
1,D:\ViSpeR\clips\videos\0YEeb-WLF2w\00001.mp4,Una tecnología blanda,82
2,D:\ViSpeR\clips\videos\0YEeb-WLF2w\00002.mp4,Las tecnologías blandas funcionan de la misma...,79
3,D:\ViSpeR\clips\videos\0YEeb-WLF2w\00003.mp4,Cuando en las historias de la Biblia se nos c...,178
4,D:\ViSpeR\clips\videos\0YEeb-WLF2w\00004.mp4,que las cosas buenas son recompensadas y las ...,110


In [5]:
print("Transcripciones vacías:", clips_df["label"].isna().sum())
print("Rutas vacías:", clips_df["path"].isna().sum())
print("Longitudes inválidas:", (clips_df["frames"] <= 0).sum())
print("Archivos inexistentes:", sum(not path.exists() for path in clips_df["path"]))

assert len(clips_df) == 1304
assert clips_df["label"].notna().all()
assert (clips_df["frames"] > 0).all()
assert all(path.exists() for path in clips_df["path"])

print("\nValidaciones superadas.")

Transcripciones vacías: 0
Rutas vacías: 0
Longitudes inválidas: 0
Archivos inexistentes: 0

Validaciones superadas.


In [7]:
all_text = "".join(clips_df["label"].astype(str))

char_counts = Counter(all_text)

vocab_df = pd.DataFrame(
    sorted(char_counts.items()),
    columns=["character", "frequency"]
)

print("Cantidad de caracteres distintos:", len(char_counts))
print("Total de caracteres:", len(all_text))

display(vocab_df)

Cantidad de caracteres distintos: 70
Total de caracteres: 92037


,character,frequency
0,,16524
1,0,57
2,1,29
3,2,29
4,3,14
...,...,...
65,í,579
66,ñ,149
67,ó,456
68,ú,86


In [8]:
special_chars = {
    char: count
    for char, count in char_counts.items()
    if not char.isalnum() and not char.isspace()
}

print("Caracteres especiales:", special_chars)
print("Espacios presentes:", " " in char_counts)
print("Mayúsculas presentes:", any(c.isupper() for c in char_counts))
print("Caracteres acentuados:", {
    c: n for c, n in char_counts.items()
    if c.isalpha() and not c.isascii()
})

Caracteres especiales: {}
Espacios presentes: True
Mayúsculas presentes: True
Caracteres acentuados: {'ó': 456, 'í': 579, 'á': 408, 'ú': 86, 'é': 327, 'ñ': 149, 'É': 1, 'ü': 1, 'Á': 5}


In [9]:
# Índice reservado para CTC
BLANK_IDX = 0

# Vocabulario determinista
characters = sorted(char_counts.keys())

char_to_idx = {
    char: idx + 1
    for idx, char in enumerate(characters)
}

idx_to_char = {
    idx: char
    for char, idx in char_to_idx.items()
}

VOCAB_SIZE = len(char_to_idx) + 1  # Incluye blank

print("Caracteres:", len(characters))
print("Tamaño del vocabulario:", VOCAB_SIZE)
print("Índice blank:", BLANK_IDX)

print("\nPrimeros índices:")
for char, idx in list(char_to_idx.items())[:15]:
    print(repr(char), "->", idx)

Caracteres: 70
Tamaño del vocabulario: 71
Índice blank: 0

Primeros índices:
' ' -> 1
'0' -> 2
'1' -> 3
'2' -> 4
'3' -> 5
'4' -> 6
'5' -> 7
'6' -> 8
'7' -> 9
'8' -> 10
'9' -> 11
'A' -> 12
'B' -> 13
'C' -> 14
'D' -> 15


In [10]:
assert BLANK_IDX not in idx_to_char
assert len(char_to_idx) == len(set(char_to_idx))
assert len(idx_to_char) == len(char_to_idx)
assert set(char_to_idx.keys()) == set(char_counts.keys())

print("Vocabulario validado.")

Vocabulario validado.


In [11]:
def encode_text(text):
    unknown_chars = set(text) - set(char_to_idx)

    if unknown_chars:
        raise ValueError(f"Caracteres fuera del vocabulario: {unknown_chars}")

    return [char_to_idx[char] for char in text]

70 caracteres + blank

In [12]:
#prueba

sample_text = clips_df.iloc[0]["label"]

encoded = encode_text(sample_text)

print("Original:", repr(sample_text))
print("Codificada:", encoded)
print("Longitud original:", len(sample_text))
print("Longitud codificada:", len(encoded))

assert len(encoded) == len(sample_text)
assert all(idx != BLANK_IDX for idx in encoded)

print("\nCodificación validada.")

Original: ' La ficción es una herramienta poderosa'
Codificada: [1, 23, 36, 1, 41, 44, 38, 38, 44, 68, 49, 1, 40, 54, 1, 56, 49, 36, 1, 43, 40, 53, 53, 36, 48, 44, 40, 49, 55, 36, 1, 51, 50, 39, 40, 53, 50, 54, 36]
Longitud original: 39
Longitud codificada: 39

Codificación validada.


In [13]:
def decode_tokens(tokens):
    return "".join(
        idx_to_char[idx]
        for idx in tokens
        if idx != BLANK_IDX
    )

In [14]:
decoded = decode_tokens(encoded)

print("Original:   ", repr(sample_text))
print("Decodificada:", repr(decoded))

assert decoded == sample_text

print("\nRound-trip validado.")

Original:    ' La ficción es una herramienta poderosa'
Decodificada: ' La ficción es una herramienta poderosa'

Round-trip validado.


finally 

In [15]:
class ViSpeRDataset(Dataset):
    def __init__(self, dataframe, image_mode):
        if image_mode not in ("grayscale", "rgb"):
            raise ValueError("image_mode debe ser 'grayscale' o 'rgb'")

        self.dataframe = dataframe.reset_index(drop=True)
        self.image_mode = image_mode

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        cap = cv2.VideoCapture(str(row["path"]))

        if not cap.isOpened():
            raise RuntimeError(f"No se pudo abrir: {row['path']}")

        frames = []

        while True:
            ret, frame = cap.read()

            if not ret:
                break

            if self.image_mode == "grayscale":
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                frame = frame[:, :, np.newaxis]
            else:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            frames.append(frame)

        cap.release()

        if len(frames) == 0:
            raise RuntimeError(f"El clip no contiene frames: {row['path']}")

        video = np.stack(frames).astype(np.float32) / 255.0

        # (T, H, W, C) -> (T, C, H, W)
        video = torch.from_numpy(video).permute(0, 3, 1, 2)

        text = str(row["label"])
        target = torch.tensor(encode_text(text), dtype=torch.long)

        return {
            "video": video,
            "target": target,
            "input_length": video.shape[0],
            "target_length": target.numel(),
            "text": text,
            "path": str(row["path"]),
        }
        

In [16]:
test_dataset = ViSpeRDataset(
    dataframe=clips_df,
    image_mode="grayscale"
)

sample = test_dataset[0]

print("Video shape:", sample["video"].shape)
print("Video dtype:", sample["video"].dtype)
print("Video min:", sample["video"].min().item())
print("Video max:", sample["video"].max().item())

print("\nTarget shape:", sample["target"].shape)
print("Input length:", sample["input_length"])
print("Target length:", sample["target_length"])
print("Texto:", repr(sample["text"]))
print("Ruta:", sample["path"])

Video shape: torch.Size([108, 1, 96, 96])
Video dtype: torch.float32
Video min: 0.019607843831181526
Video max: 1.0

Target shape: torch.Size([39])
Input length: 108
Target length: 39
Texto: ' La ficción es una herramienta poderosa'
Ruta: D:\ViSpeR\clips\videos\0YEeb-WLF2w\00000.mp4


$$
(T,H,W,C)\rightarrow(T,C,H,W)
$$

In [17]:
assert sample["video"].ndim == 4
assert sample["video"].shape[1:] == (1, 96, 96)
assert sample["video"].dtype == torch.float32

assert 0.0 <= sample["video"].min()
assert sample["video"].max() <= 1.0

assert sample["input_length"] == sample["video"].shape[0]
assert sample["target_length"] == len(sample["text"])
assert decode_tokens(sample["target"].tolist()) == sample["text"]

print("Todas las validaciones de la muestra pasaron ok")

Todas las validaciones de la muestra pasaron ok


padding temporal por batch

In [18]:
def collate_fn(batch):
    videos = [item["video"] for item in batch]
    targets = [item["target"] for item in batch]

    input_lengths = torch.tensor(
        [video.shape[0] for video in videos],
        dtype=torch.long
    )

    target_lengths = torch.tensor(
        [target.numel() for target in targets],
        dtype=torch.long
    )

    # Longitud temporal máxima del batch
    max_frames = input_lengths.max().item()

    # (B, T_max, C, H, W)
    padded_videos = torch.zeros(
        len(batch),
        max_frames,
        *videos[0].shape[1:],
        dtype=torch.float32
    )

    for i, video in enumerate(videos):
        padded_videos[i, :video.shape[0]] = video

    # CTC espera los targets concatenados
    concatenated_targets = torch.cat(targets)

    return {
        "videos": padded_videos,
        "targets": concatenated_targets,
        "input_lengths": input_lengths,
        "target_lengths": target_lengths,
        "texts": [item["text"] for item in batch],
        "paths": [item["path"] for item in batch],
    }

In [19]:
print(collate_fn)

<function collate_fn at 0x0000025631A265F0>


test con 4 clips

In [20]:
test_samples = [
    test_dataset[i]
    for i in range(4)
]

batch = collate_fn(test_samples)

print("Videos:", batch["videos"].shape)
print("Targets:", batch["targets"].shape)
print("Input lengths:", batch["input_lengths"])
print("Target lengths:", batch["target_lengths"])
print("Textos:", batch["texts"])

Videos: torch.Size([4, 178, 1, 96, 96])
Targets: torch.Size([229])
Input lengths: tensor([108,  82,  79, 178])
Target lengths: tensor([ 39,  22,  53, 115])
Textos: [' La ficción es una herramienta poderosa', ' Una tecnología blanda', ' Las tecnologías blandas funcionan de la misma manera', ' Cuando en las historias de la Biblia se nos cuenta que hay cosas que son correctas y hay cosas que son incorrectas']


los batch tienen 178 frames porque es la secuencia mas larga ( de estos 4 )

In [21]:
print("Max global:", clips_df["frames"].max())
print("Max del batch:", batch["input_lengths"].max().item())

Max global: 626
Max del batch: 178


In [22]:
# Verificar que el padding sea cero
for i, length in enumerate(batch["input_lengths"]):
    length = length.item()

    assert torch.all(batch["videos"][i, length:] == 0)

# Verificar que los frames originales se conservaron
for i, sample_item in enumerate(test_samples):
    length = sample_item["input_length"]

    assert torch.equal(
        batch["videos"][i, :length],
        sample_item["video"]
    )

# Verificar reconstrucción de cada target
offset = 0

for text, length in zip(batch["texts"], batch["target_lengths"]):
    length = length.item()
    target = batch["targets"][offset:offset + length]

    assert decode_tokens(target.tolist()) == text

    offset += length

assert offset == len(batch["targets"])

print("Padding validado.")
print("Frames originales conservados.")
print("Targets reconstruidos correctamente.")

Padding validado.
Frames originales conservados.
Targets reconstruidos correctamente.


# Crear el dataset 


In [23]:
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS
)

print("Cantidad de batches:", len(test_loader))

Cantidad de batches: 326


In [24]:
batch = next(iter(test_loader))

print("Videos:", batch["videos"].shape)
print("Targets:", batch["targets"].shape)
print("Input lengths:", batch["input_lengths"])
print("Target lengths:", batch["target_lengths"])

print("\nPrimeras transcripciones:")
for text in batch["texts"]:
    print(repr(text))

Videos: torch.Size([4, 210, 1, 96, 96])
Targets: torch.Size([327])
Input lengths: tensor([131, 210, 151,  96])
Target lengths: tensor([ 63, 123,  73,  68])

Primeras transcripciones:
' Y hasta somos capaces de enviar nuestra saliva por correo para'
'Esta afirmación me llamó la atención  me quedé pensando Luego la he escuchado más veces Quizá alguno de ustedes lo recuerde'
' pensamos en un libro contable donde están registrados los ingresos y los'
' y yo pensé y dije bueno si todo es posible qué es imposible para mí'


In [23]:
# restriccion temporal de CTC

In [25]:
def ctc_minimum_length(text):
    repeated_chars = sum(
        text[i] == text[i - 1]
        for i in range(1, len(text))
    )

    return len(text) + repeated_chars

In [26]:
clips_df["ctc_min_frames"] = clips_df["label"].apply(
    ctc_minimum_length
)

clips_df["repeated_pairs"] = (
    clips_df["ctc_min_frames"] - clips_df["label"].str.len()
)

print("Mínimo CTC:")
display(clips_df["ctc_min_frames"].describe())

print("\nPares repetidos consecutivos:")
display(clips_df["repeated_pairs"].describe())

print(
    "\nTranscripción con mayor requisito CTC:",
    repr(clips_df.loc[clips_df["ctc_min_frames"].idxmax(), "label"])
)

print(
    "Máximo requisito temporal:",
    clips_df["ctc_min_frames"].max()
)

Mínimo CTC:


count    1304.000000
mean       70.950920
std        33.060177
min         8.000000
25%        47.000000
50%        65.000000
75%        89.000000
max       292.000000
Name: ctc_min_frames, dtype: float64


Pares repetidos consecutivos:


count    1304.000000
mean        0.370399
std         0.631786
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max         5.000000
Name: repeated_pairs, dtype: float64


Transcripción con mayor requisito CTC: ' Hago esta analogía porque como yo veo la vida ahora que ya no soy tan joven es que estos mundos en mi caso han sido que voy cambiando de paradigmas o de temas y lo que me he encontrado es que en ese cambio de paradigmas o de temas a veces te encuentras en ese espacio que le llaman el vacío'
Máximo requisito temporal: 292


In [27]:
clips_df["temporal_margin"] = (
    clips_df["frames"] - clips_df["ctc_min_frames"]
)

clips_df["min_temporal_ratio"] = (
    clips_df["ctc_min_frames"] / clips_df["frames"]
)

print("Clips que no cumplen el mínimo ni siquiera antes del encoder:",
      (clips_df["temporal_margin"] < 0).sum())

print("\nMargen temporal:")
display(clips_df["temporal_margin"].describe())

print("\nProporción mínima de pasos que deben conservarse:")
display(clips_df["min_temporal_ratio"].describe())

print("\nClip con menor margen:")
display(
    clips_df.loc[
        clips_df["temporal_margin"].idxmin(),
        ["label", "frames", "ctc_min_frames", "temporal_margin"]
    ]
)

Clips que no cumplen el mínimo ni siquiera antes del encoder: 14

Margen temporal:


count    1304.000000
mean       51.358896
std        36.563615
min       -12.000000
25%        26.000000
50%        43.000000
75%        68.000000
max       334.000000
Name: temporal_margin, dtype: float64


Proporción mínima de pasos que deben conservarse:


count    1304.000000
mean        0.599578
std         0.164651
min         0.080000
25%         0.495868
50%         0.592035
75%         0.698854
max         1.200000
Name: min_temporal_ratio, dtype: float64


Clip con menor margen:


label               Porque vieron que yo les decía que dar genera...
frames                                                            89
ctc_min_frames                                                   101
temporal_margin                                                  -12
Name: 447, dtype: object

In [28]:
problematic_clips = clips_df[
    clips_df["temporal_margin"] < 0
].copy()

print("Cantidad de clips problemáticos:", len(problematic_clips))

display(
    problematic_clips[
        ["path", "label", "frames", "ctc_min_frames", "temporal_margin"]
    ]
)

Cantidad de clips problemáticos: 14


,path,label,frames,ctc_min_frames,temporal_margin
24,D:\ViSpeR\clips\videos\0YEeb-WLF2w\00024.mp4,Esto es lo que hemos visto en las películas y...,50,60,-10
42,D:\ViSpeR\clips\videos\0YEeb-WLF2w\00042.mp4,Y en esa época se estaba empezando a hablar d...,61,70,-9
71,D:\ViSpeR\clips\videos\0YEeb-WLF2w\00071.mp4,pueden servir para entender lo que no somos c...,63,64,-1
300,D:\ViSpeR\clips\videos\kOq_GFTRWxg\00013.mp4,en un momento dado y esto lo podéis ver en In...,70,78,-8
303,D:\ViSpeR\clips\videos\kOq_GFTRWxg\00016.mp4,Ayer por la noche vi que la capitalización bu...,54,63,-9
435,D:\ViSpeR\clips\videos\NNPH7KWjrrs\00043.mp4,Mientras que estábamos quizás acostumbrados a...,58,60,-2
439,D:\ViSpeR\clips\videos\NNPH7KWjrrs\00047.mp4,Y en este pequeño departamento en el que viví...,46,50,-4
447,D:\ViSpeR\clips\videos\NNPH7KWjrrs\00055.mp4,Porque vieron que yo les decía que dar genera...,89,101,-12
458,D:\ViSpeR\clips\videos\NNPH7KWjrrs\00066.mp4,ni mi hermano ni mi hermana para bancarme eso...,117,127,-10
468,D:\ViSpeR\clips\videos\NNPH7KWjrrs\00076.mp4,Y vos dirás vino a raíces Qué tiene que ver c...,72,73,-1


In [29]:
print("Proporción mínima de pasos que deben conservarse:")
display(clips_df["min_temporal_ratio"].describe())

Proporción mínima de pasos que deben conservarse:


count    1304.000000
mean        0.599578
std         0.164651
min         0.080000
25%         0.495868
50%         0.592035
75%         0.698854
max         1.200000
Name: min_temporal_ratio, dtype: float64

In [30]:
problematic_clips = problematic_clips.copy()

problematic_clips["duration_seconds"] = (
    problematic_clips["frames"] / 25
)

problematic_clips["chars_per_second"] = (
    problematic_clips["label"].str.len()
    / problematic_clips["duration_seconds"]
)

problematic_clips["video_id"] = (
    problematic_clips["path"].apply(lambda p: p.parent.name)
)

display(
    problematic_clips[
        [
            "video_id",
            "path",
            "frames",
            "duration_seconds",
            "label",
            "ctc_min_frames",
            "temporal_margin",
            "chars_per_second"
        ]
    ]
)

,video_id,path,frames,duration_seconds,label,ctc_min_frames,temporal_margin,chars_per_second
24,0YEeb-WLF2w,D:\ViSpeR\clips\videos\0YEeb-WLF2w\00024.mp4,50,2.00,Esto es lo que hemos visto en las películas y...,60,-10,30.000000
42,0YEeb-WLF2w,D:\ViSpeR\clips\videos\0YEeb-WLF2w\00042.mp4,61,2.44,Y en esa época se estaba empezando a hablar d...,70,-9,28.688525
71,0YEeb-WLF2w,D:\ViSpeR\clips\videos\0YEeb-WLF2w\00071.mp4,63,2.52,pueden servir para entender lo que no somos c...,64,-1,25.396825
300,kOq_GFTRWxg,D:\ViSpeR\clips\videos\kOq_GFTRWxg\00013.mp4,70,2.80,en un momento dado y esto lo podéis ver en In...,78,-8,27.857143
303,kOq_GFTRWxg,D:\ViSpeR\clips\videos\kOq_GFTRWxg\00016.mp4,54,2.16,Ayer por la noche vi que la capitalización bu...,63,-9,29.166667
435,NNPH7KWjrrs,D:\ViSpeR\clips\videos\NNPH7KWjrrs\00043.mp4,58,2.32,Mientras que estábamos quizás acostumbrados a...,60,-2,25.862069
439,NNPH7KWjrrs,D:\ViSpeR\clips\videos\NNPH7KWjrrs\00047.mp4,46,1.84,Y en este pequeño departamento en el que viví...,50,-4,27.173913
447,NNPH7KWjrrs,D:\ViSpeR\clips\videos\NNPH7KWjrrs\00055.mp4,89,3.56,Porque vieron que yo les decía que dar genera...,101,-12,28.089888
458,NNPH7KWjrrs,D:\ViSpeR\clips\videos\NNPH7KWjrrs\00066.mp4,117,4.68,ni mi hermano ni mi hermana para bancarme eso...,127,-10,27.136752
468,NNPH7KWjrrs,D:\ViSpeR\clips\videos\NNPH7KWjrrs\00076.mp4,72,2.88,Y vos dirás vino a raíces Qué tiene que ver c...,73,-1,25.347222


In [31]:
for idx, row in problematic_clips.iterrows():
    print(f"\nClip: {row['path'].name}")
    print(f"Video: {row['video_id']}")
    print(f"Duración: {row['duration_seconds']:.2f} s")
    print(f"Frames: {row['frames']}")
    print(f"Caracteres: {len(row['label'])}")
    print(f"Palabras: {len(row['label'].split())}")
    print(f"Requisito CTC: {row['ctc_min_frames']}")
    print(f"Transcripción: {repr(row['label'])}")
    print("-" * 80)


Clip: 00024.mp4
Video: 0YEeb-WLF2w
Duración: 2.00 s
Frames: 50
Caracteres: 60
Palabras: 13
Requisito CTC: 60
Transcripción: ' Esto es lo que hemos visto en las películas y en los libros'
--------------------------------------------------------------------------------

Clip: 00042.mp4
Video: 0YEeb-WLF2w
Duración: 2.44 s
Frames: 61
Caracteres: 70
Palabras: 14
Requisito CTC: 70
Transcripción: ' Y en esa época se estaba empezando a hablar del Internet de las cosas'
--------------------------------------------------------------------------------

Clip: 00071.mp4
Video: 0YEeb-WLF2w
Duración: 2.52 s
Frames: 63
Caracteres: 64
Palabras: 11
Requisito CTC: 64
Transcripción: ' pueden servir para entender lo que no somos capaces de entender'
--------------------------------------------------------------------------------

Clip: 00013.mp4
Video: kOq_GFTRWxg
Duración: 2.80 s
Frames: 70
Caracteres: 78
Palabras: 15
Requisito CTC: 78
Transcripción: ' en un momento dado y esto lo podéis ver en Internet 

In [32]:
clips_df["duration_seconds"] = clips_df["frames"] / 25

clips_df["words_per_second"] = (
    clips_df["label"].str.split().str.len()
    / clips_df["duration_seconds"]
)

print("Velocidad de habla estimada (palabras/s):")
display(clips_df["words_per_second"].describe())

print("\nPercentiles:")
display(
    clips_df["words_per_second"].quantile(
        [0.50, 0.75, 0.90, 0.95, 0.99, 1.00]
    )
)

print("\nRelación entre requisito CTC y frames:")
display(clips_df["min_temporal_ratio"].describe())

Velocidad de habla estimada (palabras/s):


count    1304.000000
mean        2.696524
std         0.842408
min         0.444444
25%         2.154289
50%         2.631579
75%         3.186900
max         6.500000
Name: words_per_second, dtype: float64


Percentiles:


0.50    2.631579
0.75    3.186900
0.90    3.773585
0.95    4.103560
0.99    5.054494
1.00    6.500000
Name: words_per_second, dtype: float64


Relación entre requisito CTC y frames:


count    1304.000000
mean        0.599578
std         0.164651
min         0.080000
25%         0.495868
50%         0.592035
75%         0.698854
max         1.200000
Name: min_temporal_ratio, dtype: float64

In [33]:
clips_df["video_id"] = clips_df["path"].apply(lambda p: p.parent.name)

clips_df["is_ctc_problematic"] = clips_df["temporal_margin"] < 0

video_stats = (
    clips_df.groupby("video_id")
    .agg(
        clips=("path", "count"),
        mean_words_per_second=("words_per_second", "mean"),
        max_words_per_second=("words_per_second", "max"),
        problematic_clips=("is_ctc_problematic", "sum"),
    )
    .sort_values("problematic_clips", ascending=False)
)

display(video_stats)

,clips,mean_words_per_second,max_words_per_second,problematic_clips
video_id,,,,
NNPH7KWjrrs,139,2.964839,5.312500,6
0YEeb-WLF2w,97,2.755132,6.500000,3
kOq_GFTRWxg,23,3.532706,5.454545,2
qKax5L0V4Cs,30,2.979997,5.612245,1
oL5ov6pUaCE,112,2.900123,6.147541,1
YYHIHEIwKGE,51,3.152617,4.831933,1
YFZ5RiOjUrc,27,2.822382,4.724409,0
qOt-XYC-ecM,82,2.502137,3.947368,0
of6m8w8Vgr4,113,2.752035,4.012346,0


In [34]:
print("Cantidad de clips según margen temporal:")

for threshold in [0, 5, 10, 15, 20, 30]:
    count = (clips_df["temporal_margin"] <= threshold).sum()
    print(f"Margen <= {threshold:2d}: {count:4d} clips")

print("\nClips con menor margen:")
display(
    clips_df.sort_values("temporal_margin")[
        [
            "video_id",
            "path",
            "frames",
            "ctc_min_frames",
            "temporal_margin",
            "words_per_second",
            "label"
        ]
    ].head(30)
)

Cantidad de clips según margen temporal:
Margen <=  0:   15 clips
Margen <=  5:   47 clips
Margen <= 10:   73 clips
Margen <= 15:  124 clips
Margen <= 20:  212 clips
Margen <= 30:  418 clips

Clips con menor margen:


,video_id,path,frames,ctc_min_frames,temporal_margin,words_per_second,label
447,NNPH7KWjrrs,D:\ViSpeR\clips\videos\NNPH7KWjrrs\00055.mp4,89,101,-12,5.056180,Porque vieron que yo les decía que dar genera...
24,0YEeb-WLF2w,D:\ViSpeR\clips\videos\0YEeb-WLF2w\00024.mp4,50,60,-10,6.500000,Esto es lo que hemos visto en las películas y...
458,NNPH7KWjrrs,D:\ViSpeR\clips\videos\NNPH7KWjrrs\00066.mp4,117,127,-10,5.128205,ni mi hermano ni mi hermana para bancarme eso...
42,0YEeb-WLF2w,D:\ViSpeR\clips\videos\0YEeb-WLF2w\00042.mp4,61,70,-9,5.737705,Y en esa época se estaba empezando a hablar d...
303,kOq_GFTRWxg,D:\ViSpeR\clips\videos\kOq_GFTRWxg\00016.mp4,54,63,-9,5.092593,Ayer por la noche vi que la capitalización bu...
300,kOq_GFTRWxg,D:\ViSpeR\clips\videos\kOq_GFTRWxg\00013.mp4,70,78,-8,5.357143,en un momento dado y esto lo podéis ver en In...
507,NNPH7KWjrrs,D:\ViSpeR\clips\videos\NNPH7KWjrrs\00115.mp4,44,50,-6,4.545455,El foco había estado puesto en nuestros objet...
439,NNPH7KWjrrs,D:\ViSpeR\clips\videos\NNPH7KWjrrs\00047.mp4,46,50,-4,4.891304,Y en este pequeño departamento en el que viví...
719,oL5ov6pUaCE,D:\ViSpeR\clips\videos\oL5ov6pUaCE\00073.mp4,65,69,-4,5.000000,el tercer día hice lo mismo que el segundo po...
1111,YYHIHEIwKGE,D:\ViSpeR\clips\videos\YYHIHEIwKGE\00027.mp4,130,133,-3,4.423077,Ella puso una denuncia por violencia de géner...


In [35]:
for threshold in [0, 5, 10, 15, 20, 30]:
    count = (clips_df["temporal_margin"] <= threshold).sum()
    percentage = count / len(clips_df) * 100
    print(f"Margen <= {threshold:2d}: {count:4d} clips ({percentage:.2f}%)")

Margen <=  0:   15 clips (1.15%)
Margen <=  5:   47 clips (3.60%)
Margen <= 10:   73 clips (5.60%)
Margen <= 15:  124 clips (9.51%)
Margen <= 20:  212 clips (16.26%)
Margen <= 30:  418 clips (32.06%)


15 clips de 1300 con margen nulo o negativo y solo 1 igual a 0   
restringe la arqui de la red a 3 cosas:   
1. el tiempo de los clips no se puede reducir mucho 
2. calcular longitudes temporales despeus del encoder, los originales antes de CTCLoss no van a servir 
3. no podemos descartar el 33% del dataset 

### Simulacion 

In [36]:
reduction_factors = [1, 2, 3, 4]

simulation_results = []

for factor in reduction_factors:
    clips_df[f"frames_after_x{factor}"] = np.ceil(
        clips_df["frames"] / factor
    ).astype(int)

    clips_df[f"margin_after_x{factor}"] = (
        clips_df[f"frames_after_x{factor}"]
        - clips_df["ctc_min_frames"]
    )

    invalid = (clips_df[f"margin_after_x{factor}"] < 0).sum()

    simulation_results.append({
        "reduction_factor": factor,
        "remaining_clips": len(clips_df) - invalid,
        "invalid_clips": invalid,
        "invalid_percentage": invalid / len(clips_df) * 100,
        "minimum_output_frames": clips_df[f"frames_after_x{factor}"].min(),
        "median_output_frames": clips_df[f"frames_after_x{factor}"].median(),
    })

reduction_df = pd.DataFrame(simulation_results)

display(reduction_df)

,reduction_factor,remaining_clips,invalid_clips,invalid_percentage,minimum_output_frames,median_output_frames
0,1,1290,14,1.073620,40,112.0
1,2,365,939,72.009202,20,56.0
2,3,67,1237,94.861963,14,38.0
3,4,30,1274,97.699387,10,28.0


una reducción temporal de 2x seria re agresiva. incompatible con el 70% de los clips 

In [37]:
import pyarrow

clips_df_to_save = clips_df.copy()
clips_df_to_save["path"] = clips_df_to_save["path"].astype(str)

clips_df_to_save.to_parquet(
    "data/clips_metadata.parquet",
    index=False
)

with open("data/vocabulary.json", "w", encoding="utf-8") as f:
    json.dump(char_to_idx, f, ensure_ascii=False, indent=2)
